# Thesis figures — manual annotation evaluation

Produces the figures derived from the manual validation sets, using the same
CSVs and column names as `manual_annotation_result_evaluation.ipynb`:

1. `shock_severity_confusion` — manual vs LLM severity (non-null pairs)  [thesis Fig. 6.1]
2. `manual_presence_accuracy` — per-feature presence agreement bar chart
3. `coma_presence_confusion` — coma binary confusion (manual vs LLM)

Adjust the paths in the *paths* cell if your filenames differ.

**Revisions (supervisor feedback, point 1: figures easier to understand at a
glance).** The confusion matrices now carry per-class counts (n) on the axes,
the total number of pairs (N) in the axis label, and both the raw count and its
within-row percentage in each cell; empty severity levels are dropped so the
matrix shows only the levels that actually occur. Axis labels are spelled out
(manual reference vs. LLM). No overall figure title is baked into the image —
each figure is described by its LaTeX `\caption`. A suggested caption is given
in the markdown cell above each figure.

Note on `none` severity: `normalize_sev` maps the string "none" to NaN, so
none/undocumented severities are excluded from the non-null severity pairs by
construction. The confusion matrix therefore covers only mild/moderate/severe.


In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "font.size": 11,
    "font.family": "serif", "axes.spines.top": False,
    "axes.spines.right": False,
})
OUTDIR = "figures"; os.makedirs(OUTDIR, exist_ok=True)

def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUTDIR, f"{name}.{ext}"), bbox_inches="tight")
    plt.close(fig); print("saved", name)


def plot_confusion(cm, labels, xlabel, ylabel, name,
                   cmap="Blues", drop_empty=False):
    # Confusion matrix: per-class n on the axes, total N in the x-label,
    # and count + within-row percentage in each cell.
    cm = np.asarray(cm, dtype=int)
    labels = list(labels)
    if drop_empty:                       # drop levels that never occur (row & col empty)
        keep = [i for i in range(len(labels))
                if cm[i, :].sum() > 0 or cm[:, i].sum() > 0]
        cm = cm[np.ix_(keep, keep)]
        labels = [labels[i] for i in keep]
    k = len(labels)
    row_tot = cm.sum(1); col_tot = cm.sum(0); N = int(cm.sum())

    fig, ax = plt.subplots(figsize=(0.95 * k + 2.8, 0.95 * k + 2.6))
    im = ax.imshow(cm, cmap=cmap)
    ax.set_xticks(range(k)); ax.set_yticks(range(k))
    ax.set_xticklabels([f"{l}\n(n={int(col_tot[j])})" for j, l in enumerate(labels)])
    ax.set_yticklabels([f"{l}\n(n={int(row_tot[i])})" for i, l in enumerate(labels)])
    ax.set_xlabel(f"{xlabel}\n(N = {N} non-null pairs)")
    ax.set_ylabel(ylabel)
    thr = cm.max() / 2 if cm.max() else 0.5
    for i in range(k):
        for j in range(k):
            c = int(cm[i, j])
            pct = 100 * c / row_tot[i] if row_tot[i] else 0
            ax.text(j, i, f"{c}\n({pct:.0f}%)", ha="center", va="center",
                    fontsize=10, color="white" if c > thr else "black")
    fig.colorbar(im, ax=ax, shrink=0.85, label="count")
    save(fig, name)


In [7]:
# paths (same as the evaluation notebook)
COMA_PATH    = "coma_manual_annotation_sample - coma_manual_annotation_sample.csv"
SHOCK_PATH   = "shock_annotation_comp2 - shock_manual_annotation_sample_new_prompt.csv"
LACTATE_PATH = "lactate_annotation_comp - lactate_validation_set.csv"

def normalize_bool(x):
    if pd.isna(x): return np.nan
    if isinstance(x, bool): return int(x)
    s = str(x).strip().lower()
    if s in {"true","1","yes","y"}: return 1
    if s in {"false","0","no","n"}: return 0
    return np.nan

def normalize_sev(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    return np.nan if s in {"","null","none","nan"} else s


## Figure 1 — Shock severity confusion matrix (non-null pairs) — thesis Fig. 6.1

*Suggested LaTeX caption:*

> Confusion matrix of shock-severity grades: manual reference annotation (rows)
> versus LLM-assigned severity (columns), over the notes where both annotators
> assigned a non-null severity (none/undocumented cases are excluded by
> construction, so only mild/moderate/severe appear). Each cell gives the pair
> count and, in parentheses, its share of the manual-reference row; per-class
> totals (n) are shown on the axes and the total number of pairs (N) beneath the
> x-axis. Diagonal cells are exact agreement; off-diagonal cells are almost all
> disagreements between adjacent grades rather than large jumps.


In [8]:
shock = pd.read_csv(SHOCK_PATH)
shock["m"] = shock["manual_shock_severity"].map(normalize_sev)
shock["l"] = shock["llm_severity"].map(normalize_sev)
order = ["none", "mild", "moderate", "severe"]
idx = {k: i for i, k in enumerate(order)}
mask = shock["m"].notna() & shock["l"].notna()
yt = shock.loc[mask, "m"].map(idx); yp = shock.loc[mask, "l"].map(idx)
cm = confusion_matrix(yt, yp, labels=[0, 1, 2, 3])

plot_confusion(cm, [o.capitalize() for o in order],
               "LLM-assigned severity", "Manual reference severity",
               "shock_severity_confusion", cmap="Blues", drop_empty=True)


saved shock_severity_confusion


## Figure 2 — Per-feature presence agreement

Exact-match presence accuracy on the non-null binary subset for each feature,
computed the same way as in the evaluation notebook.

*Suggested LaTeX caption:*

> Exact-match agreement between the manual reference annotation and the LLM
> presence output for each feature, on the non-null binary subset (present vs.
> not present). Each bar is annotated with the agreement rate and the number of
> annotated notes (n); the dashed line marks perfect agreement. The y-axis is
> truncated at 0.80 so that the small between-feature differences remain visible.


In [9]:
def binary_acc(path, manual_col, llm_col="llm_present"):
    df = pd.read_csv(path)
    yt = df[manual_col].map(normalize_bool)
    yp = df[llm_col].map(normalize_bool)
    m = yt.notna() & yp.notna()
    return accuracy_score(yt[m].astype(int), yp[m].astype(int)), int(m.sum())

acc_coma, n_coma   = binary_acc(COMA_PATH, "manual_coma_present")
acc_shock, n_shock = binary_acc(SHOCK_PATH, "manual_shock_present")
acc_lac, n_lac     = binary_acc(LACTATE_PATH, "human_present")

feats  = ["Lactate", "Shock", "Coma"]
accs   = [acc_lac, acc_shock, acc_coma]
ns     = [n_lac, n_shock, n_coma]
colors = ["#4C78A8", "#55A868", "#C44E52"]

fig, ax = plt.subplots(figsize=(6, 4.2))
bars = ax.bar(feats, accs, color=colors, alpha=0.9, width=0.6)
for b, a, n in zip(bars, accs, ns):
    ax.text(b.get_x() + b.get_width() / 2, a + 0.006,
            f"{a:.3f}\n(n={n})", ha="center", va="bottom", fontsize=10)

ax.axhline(1.0, color="gray", linestyle="--", linewidth=1)
ax.set_ylabel("Presence exact-match accuracy\n(manual vs. LLM, non-null subset)")
# floor adapts so every bar stays visible; capped at 0.80 when all are high
floor = min(0.80, max(0.0, np.floor((min(accs) - 0.05) * 10) / 10))
ax.set_ylim(floor, 1.10)
ax.set_yticks(np.round(np.arange(floor, 1.001, 0.05), 2))
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
save(fig, "manual_presence_accuracy")


saved manual_presence_accuracy


## Figure 3 — Coma binary confusion matrix

*Suggested LaTeX caption:*

> Confusion matrix for binary coma presence: manual reference annotation (rows)
> versus LLM output (columns), over the annotated notes with a non-null pair.
> Each cell gives the count and its share of the manual-reference row; per-class
> totals (n) are on the axes and the total (N) beneath the x-axis. Off-diagonal
> cells are the disagreements between the two annotators.


In [10]:
coma = pd.read_csv(COMA_PATH)
yt = coma["manual_coma_present"].map(normalize_bool)
yp = coma["llm_present"].map(normalize_bool)
m = yt.notna() & yp.notna()
cm = confusion_matrix(yt[m].astype(int), yp[m].astype(int), labels=[0, 1])

plot_confusion(cm, ["Not present", "Present"],
               "LLM output", "Manual reference",
               "coma_presence_confusion", cmap="Greens", drop_empty=False)


saved coma_presence_confusion
